# Table S2 — secondary and safety outcomes

**The rest of the outcomes, in one table.** `T10` and `T11` merged, keeping the columns
a reader acts on: the denominator, the weighted risk difference, the model-assisted risk difference
where a nuisance model was supported, the marginal odds ratio, and both p-values. Safety rows are
labelled descriptive in the table itself.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The rows

Straight from `report.binary_rows`, family by family. The columns beyond the estimates — the
augmentation path and the nuisance-model covariates — belong in the supplement, so they are dropped
here and nowhere else.

In [ ]:
KEPT = 14        # every column up to and including the augmented interval

def frame(family):
    rows, clauses = report.binary_rows(bundle, family)
    header = [c[:KEPT] for c in rows[:1]][0]
    return pd.DataFrame([r[:KEPT] for r in rows[1:]], columns=list(header)), clauses


secondary, secondary_clauses = frame("secondary")
safety, safety_clauses = frame("safety")

# One header, three groups of intervals: name them so the columns stay readable apart.
columns = ["outcome", "label", "denominator", "RD", "RD ci lo", "RD ci hi", "p raw", "p adjusted",
           "odds ratio", "OR ci lo", "OR ci hi", f"augmented ({report.MODEL_ASSISTED})",
           "augmented ci lo", "augmented ci hi"]
table = pd.concat([secondary, safety], ignore_index=True).set_axis(columns, axis=1)
table

## 3. Write it

In [ ]:
path = M.save_table("Table_S2", table, (
    "Table S2. Secondary and safety outcomes in the [§7] overlap population: weighted risk "
    "differences and marginal odds ratios with 95% percentile intervals, Benjamini-Hochberg "
    "correction within each family, and the model-assisted augmented estimate where the minority "
    "cell supported a nuisance model. Each estimate carries its own denominator [§11].",
    *secondary_clauses,
    *safety_clauses,
    M.provenance_clause(bundle)))

print(path)
pd.read_csv(path, comment="#")[["outcome", "label", "RD", "p raw", "p adjusted"]]

## 4. Reading it

- **`label` is the column that governs the reading.** `secondary` rows were tested; `descriptive` rows
  were estimated `[§10, §13]`.
- **Denominators differ between rows** and are printed per row `[§11]` — one outcome is recorded on
  fewer patients than the rest.
- **An odds ratio marked corrected** carries a continuity correction because a cell was empty; the
  supplement's `T10`/`T11` carry the path that produced it.